In [1]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [ ]:
region_list = ["ARK-NZK","Noord-Westelijke Delta"]

In [2]:
#For lowerlying sections

region_list = ["ARK-NZK","Noord-Westelijke Delta"]

lowerlying_dir_edited = Path(r"P:\bovenregionale-stresstest-hwn\Data\Lowerlying_sections\processed\merged_lower_lying_edited.gpkg")
merged_lower_lying_edited = gpd.read_file(lowerlying_dir_edited)

merged_lower_lying_edited = gpd.read_file(lowerlying_dir_edited)

# Buffer only the sides with 2m (excluding the ends for LineString geometries)
buffer_distance = 4  # meters

def buffer_sides_only(geometry):
    """Buffer only the sides of geometries, excluding endpoints for LineStrings"""
    if geometry.geom_type == 'LineString':
        # For LineStrings, buffer normally but this will include ends
        # To exclude ends, we can use a cap_style parameter
        return geometry.buffer(buffer_distance, cap_style=2)  # cap_style=2 is flat/square
    elif geometry.geom_type == 'MultiLineString':
        # Apply the same logic to each LineString in the MultiLineString
        from shapely.geometry import MultiPolygon
        buffered_parts = [line.buffer(buffer_distance, cap_style=2) for line in geometry.geoms]
        return MultiPolygon(buffered_parts) if len(buffered_parts) > 1 else buffered_parts[0]
    else:
        # For other geometry types (Polygon, etc.), use regular buffer
        return geometry.buffer(buffer_distance)

# Apply side-only buffering
merged_lower_lying_edited['geometry'] = merged_lower_lying_edited['geometry'].apply(buffer_sides_only)


for region in region_list:
    print(f"Processing region: {region} lowerlying entrances")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Damages_Filtering_all_columns.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)

    # Use only segments with partial lowerlying percentage
    allowed_lowerlying_only = roads_ex_gdf[
        (roads_ex_gdf['lowerlying_percentage'] > 0) & (roads_ex_gdf['lowerlying_percentage'] <= 100)
    ].copy()

    # Drop invalid/empty geometries
    allowed_lowerlying_only = allowed_lowerlying_only[
        allowed_lowerlying_only.geometry.notna() & ~allowed_lowerlying_only.geometry.is_empty
    ].copy()

    # Remove the column from left DataFrame to avoid suffixes
    if 'flooded_lowerlying_entrance' in allowed_lowerlying_only.columns:
        allowed_lowerlying_only = allowed_lowerlying_only.drop(columns=['flooded_lowerlying_entrance'])

    # Ensure CRS match (should already match)
    # Keep only the flag and geometry from roads, coerce to 0/1
    if 'flooded_lowerlying_entrance' not in roads_ex_gdf.columns:
        raise KeyError("Column 'flooded_lowerlying_entrance' not found in roads_ex_gdf")
    roads_flags = roads_ex_gdf[['flooded_lowerlying_entrance', 'geometry']].copy()
    roads_flags['flooded_lowerlying_entrance'] = (
        roads_flags['flooded_lowerlying_entrance'].fillna(0).astype(float).gt(0).astype(int)
    )

    # Spatial join: lowerlying vs roads
    joined = gpd.sjoin(
        allowed_lowerlying_only,
        roads_flags,
        how='left',
        predicate='intersects'
    )

    # Aggregate per segment: flooded if any intersecting road has flag == 1
    flooded_by_lowerlying = (
        joined.groupby(joined.index)['flooded_lowerlying_entrance']
        .max()
        .reindex(allowed_lowerlying_only.index)
        .fillna(0)
        .astype(int)
    )

    # Add the flooded flag (per segment)
    lowerlying_out = allowed_lowerlying_only.copy()
    lowerlying_out['flooded'] = flooded_by_lowerlying.values

    # Save per-segment result
    output_gpkg = root_dir.joinpath("allowed_lowerlying_flooded.gpkg")
    if output_gpkg.exists():
        output_gpkg.unlink()
    lowerlying_out.to_file(output_gpkg, driver="GPKG")
    print(f"Saved: {output_gpkg} | flooded={(lowerlying_out['flooded']==1).sum()} / {len(lowerlying_out)}")

    # ---- Dissolve segments that touch OR are within 1 m ----
    TOLERANCE_M = 1.0  # meters

    # Clean index for spatial index bookkeeping
    lowerlying_out = lowerlying_out.reset_index(drop=True)

    # Skip if empty
    if len(lowerlying_out) == 0 or lowerlying_out.geometry.is_empty.all():
        print("No lowerlying features to dissolve.")
        continue

    # Build proximity connectivity using buffered queries (distance <= TOLERANCE_M)
    try:
        buffered = lowerlying_out.geometry.buffer(TOLERANCE_M)
        idx_src, idx_tgt = lowerlying_out.sindex.query_bulk(buffered, predicate='intersects')
    except Exception as e:
        print(f"Spatial index query failed: {e}")
        idx_src = idx_tgt = []

    # Union-Find for connected components
    n = len(lowerlying_out)
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    # Add edges (skip self-pairs)
    for a, b in zip(idx_src, idx_tgt):
        if a != b:
            if b < a:
                a, b = b, a
            union(int(a), int(b))

    # Component ids
    roots = [find(i) for i in range(n)]
    unique_roots = {r: i for i, r in enumerate(sorted(set(roots)))}
    lowerlying_out['comp_id'] = [unique_roots[r] for r in roots]

    # Aggregate attributes: flooded=max; others=first
    agg = {c: 'first' for c in lowerlying_out.columns if c not in ['geometry', 'flooded', 'comp_id']}
    agg['flooded'] = 'max'

    lowerlying_out_diss = lowerlying_out.dissolve(by='comp_id', aggfunc=agg).reset_index(drop=True)

    # Save dissolved result
    output_gpkg_diss = root_dir.joinpath("allowed_lowerlying_flooded_dissolved.gpkg")
    if output_gpkg_diss.exists():
        output_gpkg_diss.unlink()
    lowerlying_out_diss.to_file(output_gpkg_diss, driver="GPKG")
    print(
        f"Saved: {output_gpkg_diss} | groups={len(lowerlying_out_diss)} | "
        f"flooded={(lowerlying_out_diss['flooded']==1).sum()}"
    )

Processing region: ARK-NZK lowerlying entrances
Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_lowerlying_flooded.gpkg | flooded=48 / 135


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_4148\736897625.py:103: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = lowerlying_out.sindex.query_bulk(buffered, predicate='intersects')


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('allowed_lowerlying_flooded_dissolved')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('allowed_lowerlying_flooded_dissolved')) failed: unable to open database file"


Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_lowerlying_flooded_dissolved.gpkg | groups=29 | flooded=13
Processing region: Noord-Westelijke Delta lowerlying entrances
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Noord-Westelijke Delta\Outputs\allowed_lowerlying_flooded.gpkg | flooded=118 / 236


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_4148\736897625.py:103: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = lowerlying_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Noord-Westelijke Delta\Outputs\allowed_lowerlying_flooded_dissolved.gpkg | groups=25 | flooded=17


In [3]:
import pandas as pd 
# add tunnels and bridge % columns to exposure and damage files and filter

data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("Tunnels_filtered_by_area_2000_th.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

kunstinweg_gdf = gpd.read_file(kunstinweg)
kunstinweg_gdf['geometry'] = kunstinweg_gdf['geometry'].buffer(0.2) # to make sure lines are valid
kunstinweg_gdf.rename(columns={'OMSCHR': 'objecttekst'}, inplace=True)

kunstinweg_bridge = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'brug']
kunstinweg_tunnel = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'tunnel']


tunnels_gdf_kunstoverweg = gpd.read_file(tunnels)
bridges_gdf_kunstoverweg = gpd.read_file(bridges) 

bridges_gdf = gpd.GeoDataFrame(
    pd.concat([bridges_gdf_kunstoverweg, kunstinweg_bridge], ignore_index=True),
    crs=bridges_gdf_kunstoverweg.crs
)

tunnels_gdf = gpd.GeoDataFrame(
    pd.concat([tunnels_gdf_kunstoverweg, kunstinweg_tunnel], ignore_index=True),
    crs=tunnels_gdf_kunstoverweg.crs
)



# Define allowed values for bridges and tunnels (lowercased for case-insensitive matching)
allowed_bridges = [
    'aanbrug', 'brug', 'brug (beweegbaar)', 'brug (landbouw)', 'brug (vast)',
    'brug beton', 'brug beton in', 'brug beton over', 'brug beweegbaar',
    'brug hout in', 'brug in', 'brug in de toerit va', 'brug staal in',
    'brug vast', 'vaste brug'
]

allowed_tunnels = [
    'cervedict tunnel', 'open tunnelbak', 'tunnel', 'tunnel vlak',
    'tunnelbak', 'tunnelbak den kaat'
]

filtered_viaducts = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'viaduct']

# Convert to lowercase for case-insensitive comparison
allowed_bridges = [x.lower() for x in allowed_bridges]
allowed_tunnels = [x.lower() for x in allowed_tunnels]

# Filter bridges
filtered_gdf_brug = bridges_gdf[
    bridges_gdf['objecttekst'].str.lower().isin(allowed_bridges)
]

# Filter tunnels
filtered_gdf_tunnel_and_bridges = tunnels_gdf[
    tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels + allowed_bridges)
]


In [9]:
#Flooded tunnel entrances
region_list = ["ARK-NZK","Vallei en Veluwe",
               "Achterhoek", "Brabantse Delta","Friesland",
               "Groningen en NO-Drenthe","Limburg",
               "Noord-Brabant Oost","Noord-Westelijke Delta",
               "Rivierenland","Scheldestromen","Zuiderzeeland",
               "Overijsselse Vecht"
               ]

for region in region_list:
    print(f"Processing region: {region} tunnels")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Damages_Filtering_all_columns.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)

    # Use only allowed tunnels (not bridges) OR tunnels with 'OPEN' in INVENT_OMS
    allowed_tunnels_only = tunnels_gdf[
        (tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels)) |
        (tunnels_gdf['INVENT_OMS'].str.upper().str.contains('OPEN', na=False))
    ].copy()

    # Drop invalid/empty geometries
    allowed_tunnels_only = allowed_tunnels_only[
        allowed_tunnels_only.geometry.notna() & ~allowed_tunnels_only.geometry.is_empty
    ].copy()

    # Ensure CRS match
    if allowed_tunnels_only.crs != roads_ex_gdf.crs:
        allowed_tunnels_only = allowed_tunnels_only.to_crs(roads_ex_gdf.crs)

    # Keep only the flag and geometry from roads, coerce to 0/1
    if 'flooded_tunnel_entrance' not in roads_ex_gdf.columns:
        raise KeyError("Column 'flooded_tunnel_entrance' not found in roads_ex_gdf")
    roads_flags = roads_ex_gdf[['flooded_tunnel_entrance', 'geometry']].copy()
    roads_flags['flooded_tunnel_entrance'] = (
        roads_flags['flooded_tunnel_entrance'].fillna(0).astype(float).gt(0).astype(int)
    )

    # Spatial join: tunnels vs roads
    joined = gpd.sjoin(
        allowed_tunnels_only,
        roads_flags,
        how='left',
        predicate='intersects'
    )

    # Aggregate per tunnel: flooded if any intersecting road has flag == 1
    flooded_by_tunnel = (
        joined.groupby(joined.index)['flooded_tunnel_entrance']
        .max()
        .reindex(allowed_tunnels_only.index)
        .fillna(0)
        .astype(int)
    )

    # Add the flooded flag (per tunnel)
    tunnels_out = allowed_tunnels_only.copy()
    tunnels_out['flooded'] = flooded_by_tunnel.values

    # Save per-tunnel result
    output_gpkg = root_dir.joinpath("allowed_tunnels_flooded.gpkg")
    if output_gpkg.exists():
        output_gpkg.unlink()
    tunnels_out.to_file(output_gpkg, driver="GPKG")
    print(f"Saved: {output_gpkg} | flooded={(tunnels_out['flooded']==1).sum()} / {len(tunnels_out)}")

    # ---- Dissolve tunnels that touch OR are within 1 m ----
    TOLERANCE_M = 1.0  # meters

    # Clean index for spatial index bookkeeping
    tunnels_out = tunnels_out.reset_index(drop=True)

    # Skip if empty
    if len(tunnels_out) == 0 or tunnels_out.geometry.is_empty.all():
        print("No tunnel features to dissolve.")
        continue

    # Build proximity connectivity using buffered queries (distance <= TOLERANCE_M)
    try:
        buffered = tunnels_out.geometry.buffer(TOLERANCE_M)
        idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')
    except Exception as e:
        print(f"Spatial index query failed: {e}")
        idx_src = idx_tgt = []

    # Union-Find for connected components
    n = len(tunnels_out)
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    # Add edges (skip self-pairs)
    for a, b in zip(idx_src, idx_tgt):
        if a != b:
            if b < a:
                a, b = b, a
            union(int(a), int(b))

    # Component ids
    roots = [find(i) for i in range(n)]
    unique_roots = {r: i for i, r in enumerate(sorted(set(roots)))}
    tunnels_out['comp_id'] = [unique_roots[r] for r in roots]

    # Aggregate attributes: flooded=max; others=first
    agg = {c: 'first' for c in tunnels_out.columns if c not in ['geometry', 'flooded', 'comp_id']}
    agg['flooded'] = 'max'
    # Also aggregate INVENT_OMS to preserve information about OPEN entrances
    if 'INVENT_OMS' in tunnels_out.columns:
        agg['INVENT_OMS'] = lambda x: '|'.join(x.dropna().astype(str))

    tunnels_out_diss = tunnels_out.dissolve(by='comp_id', aggfunc=agg).reset_index(drop=True)

    # Additional logic: If any tunnel group contains 'OPEN' in INVENT_OMS, check flood depth criteria
    if 'INVENT_OMS' in tunnels_out_diss.columns:
        for idx, row in tunnels_out_diss.iterrows():
            has_open = 'OPEN' in str(row['INVENT_OMS']).upper()
            if has_open:
                # Get individual tunnels in this group
                group_tunnels = tunnels_out[tunnels_out['comp_id'] == idx]
                
                # Spatial join with roads to get flood depth values
                group_roads_join = gpd.sjoin(
                    group_tunnels,
                    roads_ex_gdf[['F_EV1_me', 'F_EV1_fr', 'geometry']],
                    how='left',
                    predicate='intersects'
                )
                
                # Check if any intersecting road meets flood criteria
                if len(group_roads_join) > 0:
                    flood_criteria = (
                        (group_roads_join['F_EV1_me'].fillna(0) >= 0.1) & 
                        (group_roads_join['F_EV1_fr'].fillna(0) >= 0.25)
                    )
                    
                    if flood_criteria.any():
                        tunnels_out_diss.loc[idx, 'flooded'] = 1
                        #print(f"Tunnel group {idx} marked as flooded due to OPEN entrance meeting flood depth criteria")

    # Save dissolved result
    output_gpkg_diss = root_dir.joinpath("allowed_tunnels_flooded_dissolved.gpkg")
    if output_gpkg_diss.exists():
        output_gpkg_diss.unlink()
    tunnels_out_diss.to_file(output_gpkg_diss, driver="GPKG")
    print(
        f"Saved: {output_gpkg_diss} | groups={len(tunnels_out_diss)} | "
        f"flooded={(tunnels_out_diss['flooded']==1).sum()}"
    )

Processing region: ARK-NZK tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_tunnels_flooded.gpkg | flooded=16 / 266


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_4148\1187499242.py:81: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=12
Processing region: Vallei en Veluwe tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Vallei en Veluwe\Outputs\allowed_tunnels_flooded.gpkg | flooded=0 / 266


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_4148\1187499242.py:81: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Vallei en Veluwe\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=0
Processing region: Achterhoek tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Achterhoek\Outputs\allowed_tunnels_flooded.gpkg | flooded=0 / 266


...

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 266 WHERE lower(table_name) = lower('allowed_tunnels_flooded')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 266 WHERE lower(table_name) = lower('allowed_tunnels_flooded')) failed: unable to open database file"
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_4148\1187499242.py:81: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Zuiderzeeland\Outputs\allowed_tunnels_flooded.gpkg | flooded=0 / 266
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Zuiderzeeland\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=0
Processing region: Overijsselse Vecht tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Overijsselse Vecht\Outputs\allowed_tunnels_flooded.gpkg | flooded=1 / 266


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_4148\1187499242.py:81: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Overijsselse Vecht\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=1


In [6]:
# Save dissolved result with error handling
output_gpkg_diss = root_dir.joinpath("allowed_tunnels_flooded_dissolved.gpkg")
max_retries = 3
for attempt in range(max_retries):
    try:
        if output_gpkg_diss.exists():
            output_gpkg_diss.unlink()
        
        # Add a small delay to ensure file is fully released
        import time
        time.sleep(0.1)
        
        # Try to save with explicit driver options
        tunnels_out_diss.to_file(
            output_gpkg_diss, 
            driver="GPKG",
            layer='allowed_tunnels_flooded_dissolved'
        )
        print(f"Saved: {output_gpkg_diss} | groups={len(tunnels_out_diss)} | flooded={(tunnels_out_diss['flooded']==1).sum()}")
        break
        
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {str(e)}")
        if attempt == max_retries - 1:
            # Try alternative format as fallback
            output_shp = root_dir.joinpath("allowed_tunnels_flooded_dissolved.shp")
            if output_shp.exists():
                output_shp.unlink()
            tunnels_out_diss.to_file(output_shp, driver="ESRI Shapefile")
            print(f"Saved as Shapefile: {output_shp} | groups={len(tunnels_out_diss)} | flooded={(tunnels_out_diss['flooded']==1).sum()}")
        else:
            time.sleep(1)  # Wait before retry

Saved: P:\bovenregionale-stresstest-hwn\Analysis\Friesland\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=3
